# Generador sintético de CUMST, DEAL y TRANS

Este notebook simula la llegada de datos de negocio a la capa Bronze del catálogo `raw_data`, creando tres tablas sintéticas: `cumst`, `deal` y `trans`.

Objetivo:
- Crear el catálogo y el esquema Bronze si no existen
- Generar 500 registros de CUMST (clientes)
- Generar 500 registros de DEAL (productos)
- Generar 1500 registros de TRANS (transacciones)
- Publicar las tres tablas en `raw_data.bronze`

In [0]:
import os
import time
import uuid
from datetime import datetime, timezone

from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOG_NAME = "raw_data"
SCHEMA_NAME = "bronze"
SOURCE_SYSTEM = "AS400_GENERIC"
CUMST_ROWS = int(os.getenv("SDP_CUMST_ROWS", 500))
DEAL_ROWS = int(os.getenv("SDP_DEAL_ROWS", 500))
TRANS_ROWS = int(os.getenv("SDP_TRANS_ROWS", 1500))
NOTEBOOK_RUNTIME_MINUTES = int(os.getenv("SDP_NOTEBOOK_RUNTIME_MINUTES", 30))
RUNTIME_MULTIPLIER = float(os.getenv("SDP_RUNTIME_MULTIPLIER", 1.0))
RUN_TS = datetime.now(timezone.utc)
BATCH_ID = f"BATCH-{RUN_TS.strftime('%Y%m%d%H%M%S')}-{uuid.uuid4().hex[:6].upper()}"

PIPELINE_STAGE_PLAN = {
    "catalog": 5,
    "cumst": 5,
    "deal": 5,
    "persist": 5,
    "validation": 3,
    "trans": 5,
    "publish": 2,
}

def run_stage(stage_name: str, duration_minutes: int, fn=None):
    total_seconds = max(1, int(duration_minutes * 60 * RUNTIME_MULTIPLIER))
    print(f"\n=== {stage_name} ===")
    print(f"Planned runtime: {duration_minutes} min | multiplier={RUNTIME_MULTIPLIER}")
    elapsed = 0
    while elapsed < total_seconds:
        sleep_for = min(15, total_seconds - elapsed)
        time.sleep(sleep_for)
        elapsed += sleep_for
        progress = min(100, round((elapsed / total_seconds) * 100, 1))
        print(f"[{stage_name}] {progress}% complete")
    if fn is not None:
        return fn()
    return None

print(f"Notebook runtime target: {NOTEBOOK_RUNTIME_MINUTES} minutes")
print(f"Stage plan: {PIPELINE_STAGE_PLAN}")

In [0]:
def create_catalog_and_schema():
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
    spark.sql(f"USE CATALOG {CATALOG_NAME}")
    spark.sql(f"USE SCHEMA {SCHEMA_NAME}")
    print(f"Catalogo listo: {CATALOG_NAME}.{SCHEMA_NAME}")
    return True

run_stage("1/7 - Crear catálogo y esquema Bronze", PIPELINE_STAGE_PLAN["catalog"], create_catalog_and_schema)

In [0]:
def build_cumst(n_rows: int):
    male_names = F.array(*[F.lit(x) for x in ["Eirik", "Leif", "Bjorn", "Andreas", "Soren", "Magnus"]])
    female_names = F.array(*[F.lit(x) for x in ["Astrid", "Freja", "Ingrid", "Sofia", "Eleni", "Mariam"]])
    nordic_last = F.array(*[F.lit(x) for x in ["Johansen", "Andersson", "Nielsen", "Svensson", "Larsen", "Hansen"]])
    greek_last = F.array(*[F.lit(x) for x in ["Papadopoulos", "Nikolaidis", "Georgiou", "Kostopoulos", "Vassiliou"]])
    palestinian_last = F.array(*[F.lit(x) for x in ["Al Khalidi", "Hamdan", "Barghouti", "Abu Salem", "Nusseibeh"]])
    no_cities = F.array(*[F.lit(x) for x in ["Oslo", "Bergen", "Trondheim"]])
    gr_cities = F.array(*[F.lit(x) for x in ["Athens", "Thessaloniki", "Patras"]])
    ps_cities = F.array(*[F.lit(x) for x in ["Ramallah", "Nablus", "Bethlehem"]])
    df = spark.range(n_rows).withColumnRenamed("id", "row_id")
    gender = F.when((F.col("row_id") % 2) == 0, F.lit("M")).otherwise(F.lit("F"))
    country = (
        F.when((F.col("row_id") % 3) == 0, F.lit("NO"))
         .when((F.col("row_id") % 3) == 1, F.lit("GR"))
         .otherwise(F.lit("PS"))
    )
    first_name = F.when(
        gender == "M",
        F.element_at(male_names, (F.pmod(F.col("row_id"), F.lit(6)) + F.lit(1)).cast("int"))
    ).otherwise(
        F.element_at(female_names, (F.pmod(F.col("row_id"), F.lit(6)) + F.lit(1)).cast("int"))
    )
    last_name = (
        F.when(country == "NO", F.element_at(nordic_last, (F.pmod(F.col("row_id"), F.lit(6)) + F.lit(1)).cast("int")))
         .when(country == "GR", F.element_at(greek_last, (F.pmod(F.col("row_id"), F.lit(5)) + F.lit(1)).cast("int")))
         .otherwise(F.element_at(palestinian_last, (F.pmod(F.col("row_id"), F.lit(5)) + F.lit(1)).cast("int")))
    )
    city = (
        F.when(country == "NO", F.element_at(no_cities, (F.pmod(F.col("row_id"), F.lit(3)) + F.lit(1)).cast("int")))
         .when(country == "GR", F.element_at(gr_cities, (F.pmod(F.col("row_id"), F.lit(3)) + F.lit(1)).cast("int")))
         .otherwise(F.element_at(ps_cities, (F.pmod(F.col("row_id"), F.lit(3)) + F.lit(1)).cast("int")))
    )
    email_domain = (
        F.when(country == "NO", F.lit("nordmail.com"))
         .when(country == "GR", F.lit("hellasmail.gr"))
         .otherwise(F.lit("levantmail.ps"))
    )
    doc_prefix = F.when(country == "NO", F.lit("NOR")).when(country == "GR", F.lit("GRC")).otherwise(F.lit("PAL"))
    phone_prefix = F.when(country == "NO", F.lit("+47")).when(country == "GR", F.lit("+30")).otherwise(F.lit("+970"))
    return (
        df
        .withColumn("customer_id", F.format_string("CUS-%05d", F.col("row_id") + F.lit(1)))
        .withColumn("gender", gender)
        .withColumn("first_name", first_name)
        .withColumn("last_name", last_name)
        .withColumn("full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
        .withColumn("country", country)
        .withColumn("city", city)
        .withColumn("document_id", F.concat(doc_prefix, F.lit("-"), F.format_string("%09d", F.col("row_id") + F.lit(100000001))))
        .withColumn("email", F.lower(F.concat_ws(".", F.regexp_replace(F.col("first_name"), " ", ""), F.regexp_replace(F.col("last_name"), " ", ""), email_domain)))
        .withColumn("phone", F.concat(phone_prefix, F.lit("-"), F.format_string("%08d", F.col("row_id") + F.lit(10000000))))
        .withColumn("birth_date", F.date_sub(F.current_date(), (F.lit(18 * 365) + (F.col("row_id") % F.lit(45)) * F.lit(365)).cast("int")))
        .withColumn("status", F.when((F.col("row_id") % 10) == 0, F.lit("I")).otherwise(F.lit("A")))
        .withColumn("source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("ingest_ts", F.lit(RUN_TS))
        .withColumn("batch_id", F.lit(BATCH_ID))
        .withColumn("ingest_date", F.to_date(F.lit(RUN_TS.date().isoformat())))
        .drop("row_id")
    )


def build_deal(cumst_df):
    customer_lookup = cumst_df.select("customer_id", F.row_number().over(Window.orderBy("customer_id")).alias("customer_rn"))
    base = spark.range(DEAL_ROWS).withColumn("rn", (F.pmod(F.col("id"), F.lit(CUMST_ROWS)) + F.lit(1)).cast("int"))
    product_code = F.array(*[F.lit(x) for x in ["LOAN", "CARD", "MORT", "SAV", "INV", "AUTO"]])
    currency = F.when((F.col("id") % 5) == 0, F.lit("USD")).otherwise(F.lit("CRC"))
    deal = (
        base
        .join(F.broadcast(customer_lookup), F.col("rn") == F.col("customer_rn"), "left")
        .withColumn("deal_id", F.format_string("DEA-%05d", F.col("id") + F.lit(1)))
        .withColumn("customer_id", F.col("customer_id"))
        .withColumn("product_code", F.element_at(product_code, (F.pmod(F.col("id"), F.lit(6)) + F.lit(1)).cast("int")))
        .withColumn("currency", currency)
        .withColumn("principal_balance", F.round((F.pow((F.col("id") % 97) + F.lit(2), F.lit(1.45)) * F.lit(123.45)).cast("double"), 2))
        .withColumn("interest_rate", F.round(F.lit(0.08) + (F.col("id") % F.lit(40)) * F.lit(0.0015), 6))
        .withColumn("open_date", F.date_sub(F.current_date(), (F.col("id") % F.lit(720)).cast("int")))
        .withColumn("maturity_date", F.date_add(F.current_date(), (F.col("id") % F.lit(900)).cast("int")))
        .withColumn("status", F.when((F.col("id") % 12) == 0, F.lit("S")).otherwise(F.lit("A")))
        .withColumn("source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("ingest_ts", F.lit(RUN_TS))
        .withColumn("batch_id", F.lit(BATCH_ID))
        .withColumn("ingest_date", F.to_date(F.lit(RUN_TS.date().isoformat())))
        .drop("rn", "customer_rn", "id")
    )
    return deal

def build_trans(cumst_df, deal_df, rows: int):
    cumst_lookup = cumst_df.select(
        F.col("customer_id").alias("cumst_customer_id"),
        F.row_number().over(Window.orderBy("customer_id")).alias("customer_rn")
    )
    deal_lookup = deal_df.select(
        F.col("deal_id").alias("deal_id_key"),
        F.col("customer_id").alias("deal_customer_id"),
        F.row_number().over(Window.orderBy("deal_id")).alias("deal_rn")
    )

    base = spark.range(rows).withColumn("row_id", (F.col("id") + F.lit(1)).cast("int"))
    trx_codes = F.array(*[F.lit(x) for x in ["DEP", "RET", "TRF", "INT", "COM", "PAY", "REV"]])

    trans = (
        base
        .withColumn("customer_rn", (F.pmod(F.col("id"), F.lit(CUMST_ROWS)) + F.lit(1)).cast("int"))
        .withColumn("deal_rn", (F.pmod(F.col("id"), F.lit(DEAL_ROWS)) + F.lit(1)).cast("int"))
        .join(F.broadcast(cumst_lookup), "customer_rn", "left")
        .join(F.broadcast(deal_lookup), "deal_rn", "left")
        .withColumn("trx_id", F.format_string("TRX-%010d", F.col("row_id") + F.lit(1000000000)))
        .withColumn("tracde", F.element_at(trx_codes, (F.pmod(F.col("id"), F.lit(7)) + F.lit(1)).cast("int")))
        .withColumn("tradcc", F.when(F.col("tracde").isin("RET", "COM", "PAY"), F.lit("D")).otherwise(F.lit("C")))
        .withColumn("customer_id", F.col("cumst_customer_id"))
        .withColumn("deal_id", F.col("deal_id_key"))
        .withColumn("tracun", F.col("cumst_customer_id"))
        .withColumn("tragln", F.regexp_replace(F.col("deal_id_key"), "DEA-", "").cast("long") + F.lit(10000000000))
        .withColumn("traacc", F.regexp_replace(F.col("deal_id_key"), "DEA-", "").cast("long") + F.lit(7000000000))
        .withColumn("trabnk", F.lit("001"))
        .withColumn("trabrn", (F.pmod(F.col("id"), F.lit(90)) + F.lit(1)).cast("int"))
        .withColumn("traccy", F.when((F.col("id") % 5) == 0, F.lit("USD")).otherwise(F.lit("CRC")))
        .withColumn("traamt", F.round((F.pow((F.pmod(F.col("id"), F.lit(180)) + F.lit(2)).cast("double"), F.lit(1.18)) * F.lit(17.35)), 2))
        .withColumn("travdm", F.date_add(F.current_date(), -(F.pmod(F.col("id"), F.lit(120))).cast("int")))
        .withColumn("trabdy", F.date_sub(F.current_date(), (F.pmod(F.col("id"), F.lit(120))).cast("int")))
        .withColumn("trausr", F.concat(F.lit("USR"), F.format_string("%03d", F.pmod(F.col("id"), F.lit(75)) + F.lit(1))))
        .withColumn("traapc", F.lit("AS400"))
        .withColumn("tramod", F.when((F.col("id") % 3) == 0, F.lit("L")).otherwise(F.lit("T")))
        .withColumn("trapmn", F.pmod(F.col("id"), F.lit(12)) + F.lit(1))
        .withColumn("trackn", F.pmod(F.col("id"), F.lit(999999)) + F.lit(1))
        .withColumn("traccn", (F.pmod(F.col("id"), F.lit(25)) + F.lit(100)).cast("int"))
        .withColumn("source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("ingest_ts", F.lit(RUN_TS))
        .withColumn("batch_id", F.lit(BATCH_ID))
        .withColumn("ingest_date", F.to_date(F.lit(RUN_TS.date().isoformat())))
        .drop("id", "row_id", "customer_rn", "deal_rn", "cumst_customer_id", "deal_id_key", "deal_customer_id")
    )
    return trans


def generate_customer_and_deal_tables():
    cumst_df = build_cumst(CUMST_ROWS)
    deal_df = build_deal(cumst_df)
    cumst_df.show(5, truncate=False)
    deal_df.show(5, truncate=False)
    return cumst_df, deal_df

cumst_df, deal_df = run_stage("2/7 - Generar CUMST y DEAL", PIPELINE_STAGE_PLAN["cumst"] + PIPELINE_STAGE_PLAN["deal"], generate_customer_and_deal_tables)

In [0]:
def preview_bronze_tables():
    print(f"CUMST rows generadas: {cumst_df.count()}")
    print(f"DEAL rows generadas: {deal_df.count()}")
    cumst_df.select("customer_id", "full_name", "country", "city").show(5, truncate=False)
    deal_df.select("deal_id", "customer_id", "currency", "principal_balance").show(5, truncate=False)
    return True

run_stage("3/7 - Preview CUMST y DEAL", PIPELINE_STAGE_PLAN['persist'], preview_bronze_tables)


## Validación rápida

Revisa los esquemas, conteos y algunas filas para confirmar que el aterrizaje a Bronze se completó correctamente.

In [0]:
def validate_bronze_quality():
    spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.cumst").select("customer_id", "full_name", "city", "country", "batch_id").show(10, truncate=False)
    spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.deal").select("deal_id", "customer_id", "currency", "principal_balance", "batch_id").show(10, truncate=False)
    print("CUMST duplicates:", spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.cumst").groupBy("customer_id").count().filter("count > 1").count())
    print("DEAL duplicates:", spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.deal").groupBy("deal_id").count().filter("count > 1").count())

    spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.trans").select("trx_id", "customer_id", "deal_id", "tracde", "tradcc", "traamt", "travdm", "batch_id").show(10, truncate=False)
    print("TRANS duplicates:", spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.trans").groupBy("trx_id").count().filter("count > 1").count())

    # Integridad referencial
    orphan_trans_cumst = spark.sql(f"SELECT COUNT(*) FROM {CATALOG_NAME}.{SCHEMA_NAME}.trans t LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.cumst c ON t.customer_id = c.customer_id WHERE c.customer_id IS NULL").collect()[0][0]
    orphan_trans_deal = spark.sql(f"SELECT COUNT(*) FROM {CATALOG_NAME}.{SCHEMA_NAME}.trans t LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.deal d ON t.deal_id = d.deal_id WHERE d.deal_id IS NULL").collect()[0][0]
    print(f"TRANS → CUMST huérfanos: {orphan_trans_cumst}")
    print(f"TRANS → DEAL huérfanos: {orphan_trans_deal}")

    return True

run_stage("4/7 - Validar calidad de Bronze", PIPELINE_STAGE_PLAN['validation'], validate_bronze_quality)


In [0]:
def generate_transaction_table():
    trans_df = build_trans(cumst_df, deal_df, TRANS_ROWS)
    trans_df.show(5, truncate=False)

    return trans_df

trans_df = run_stage("5/7 - Generar transacciones", PIPELINE_STAGE_PLAN['trans'], generate_transaction_table)


## Generar y publicar TRANS

En esta sección se construyen las transacciones sintéticas usando CUMST y DEAL como base y se deja lista la tabla `raw_data.bronze.trans` antes de la publicación final.

## Publicar en raw_data

En esta última parte del mismo notebook se crea el catálogo `raw_data`, el esquema `bronze` y se publican las tablas sintéticas directamente en Bronze. Así, con una sola ejecución cubres generación y aterrizaje.

In [0]:
RAW_CATALOG = "raw_data"
RAW_SCHEMA = "bronze"

def finalize_bronze_publish():
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {RAW_CATALOG}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {RAW_CATALOG}.{RAW_SCHEMA}")
    spark.sql(f"USE CATALOG {RAW_CATALOG}")
    spark.sql(f"USE SCHEMA {RAW_SCHEMA}")
    (
        cumst_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{RAW_CATALOG}.{RAW_SCHEMA}.cumst")
    )
    (
        deal_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{RAW_CATALOG}.{RAW_SCHEMA}.deal")
    )
    (
        trans_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{RAW_CATALOG}.{RAW_SCHEMA}.trans")
    )
    print("Publicado en raw_data.bronze.cumst:", spark.table(f"{RAW_CATALOG}.{RAW_SCHEMA}.cumst").count())
    print("Publicado en raw_data.bronze.deal:", spark.table(f"{RAW_CATALOG}.{RAW_SCHEMA}.deal").count())
    print("Publicado en raw_data.bronze.trans:", spark.table(f"{RAW_CATALOG}.{RAW_SCHEMA}.trans").count())
    return True

run_stage("6/7 - Publicar tabla final en Bronze", PIPELINE_STAGE_PLAN['publish'], finalize_bronze_publish)